<a href="https://colab.research.google.com/github/senchiao/HRRR_plots/blob/main/surface_stn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [163]:
!pip cache purge
!pip uninstall -y metpy cartopy
!pip install metpy==1.7
!pip install cartopy==0.16.0

import metpy
import cartopy

import pandas as pd
from datetime import datetime, timedelta
from io import StringIO
from urllib.request import urlopen


from metpy.io import metar
import metpy.plots as mpplots
from metpy.units import units
import matplotlib.pyplot as plt

import cartopy.crs as ccrs

print(f"MetPy version: {metpy.__version__}")
print(f"Cartopy version: {cartopy.__version__}")

Files removed: 10
Found existing installation: MetPy 1.7.0
Uninstalling MetPy-1.7.0:
  Successfully uninstalled MetPy-1.7.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 424.3/424.3 kB 5.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 48.5 MB/s eta 0:00:00
  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (setup.py) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.
MetPy version: 1.7.1
Cartopy version: 0.25.0


In [164]:
date = datetime(2026, 8, 5, 10)

# Remote Access - Archive Data Read with pandas from Iowa State Archive,
# note differences from METAR files
dt = timedelta(minutes=15)
sdate = date - dt
edate = date + dt
data_url = ('http://mesonet.agron.iastate.edu/cgi-bin/request/asos.py?'
            'data=all&tz=Etc/UTC&format=comma&latlon=yes&'
            f'year1={sdate.year}&month1={sdate.month}&day1={sdate.day}'
            f'&hour1={sdate.hour}&minute1={sdate.minute}&'
            f'year2={edate.year}&month2={edate.month}&day2={edate.day}'
            f'&hour2={edate.hour}&minute2={edate.minute}')
data = pd.read_csv(data_url, skiprows=5, na_values=['M'],
                   low_memory=False).replace('T', 0.00001).groupby('station').tail(1)
df = metar.parse_metar_file(StringIO('\n'.join(val for val in data.metar)),
                            year=date.year, month=date.month)
df['date_time'] = date

df['tmpf'] = (df.air_temperature.values * units.degC).to('degF')
df['dwpf'] = (df.dew_point_temperature.values * units.degC).to('degF')

In [166]:
mslp_formatter = lambda v: format(v*10, '.0f')[-3:]

# Plot desired data
obs = declarative.PlotObs()
obs.data = df
obs.time = date
obs.time_window = timedelta(minutes=15)
obs.level = None
obs.fields = ['cloud_coverage', 'tmpf', 'dwpf',
              'air_pressure_at_sea_level', 'current_wx1_symbol']
obs.locations = ['C', 'NW', 'SW', 'NE', 'W']
obs.formats = ['sky_cover', None, None, mslp_formatter, 'current_weather']
obs.reduce_points = 0.15
obs.vector_field = ['eastward_wind', 'northward_wind']

# Panel for plot with Map features
panel = declarative.MapPanel()
panel.layout = (1, 1, 1)
panel.projection = 'lcc'

panel.area = 'MO' # Removed as it causes ImportError
panel.layers = ['states']
panel.plots = [obs]

# Bringing it all together
plt = declarative.PanelContainer()
plt.size = (20, 20)
plt.panels = [panel]

plt.show()


ImportError: cannot import name 'named_areas' from 'metpy.plots.plot_areas' (/usr/local/lib/python3.13/dist-packages/metpy/plots/plot_areas.py)